# Two Maps — healthy life expectancy vs RTT waiting times · **Bronze layer**

**What this notebook does, in one breath:** pulls the five raw sources down
unmodified and stamps each with where it came from, when, and a SHA-256, so
every number that reaches the article can be traced back to a file I can
re-download and re-hash.

The point of the whole project is to test a claim before writing it: that
healthy life expectancy (published by *where you live*, MSOA) and RTT waiting
times (published by *where you're treated*, provider) cannot be cleanly joined.
Bronze doesn't test anything. It just makes sure the inputs are what I say
they are.

The pipeline logic lives in `src/pipeline/`; each notebook runs the relevant
steps and then inspects what they produced, so there is one source of truth
for the code and a readable trail of what it did.

## Step 0 — Tools

In [1]:
import importlib.util, sys
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PIPELINE = PROJECT_DIR / "src" / "pipeline"
sys.path.insert(0, str(PIPELINE))
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 120)

def run_step(name: str):
    """Import and run one pipeline step by filename (they start with digits, so importlib)."""
    spec = importlib.util.spec_from_file_location(name, PIPELINE / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.main()
    return mod

BRONZE = PROJECT_DIR / "data" / "bronze"
SILVER = PROJECT_DIR / "data" / "silver"
GOLD = PROJECT_DIR / "data" / "gold"
print(PROJECT_DIR)

/Users/yusufismail/Two_maps


## Step 1 — The five sources

| # | Source | Why it's here |
|---|---|---|
| 1 | ONS, *Health state life expectancy estimates for MSOAs, E&W: 2019 to 2023* (30 Jul 2026) | The origin of the HLE figures. |
| 2 | OHID Local health (Fingertips), indicator 94359 (2 Sep 2026) | The redistribution the article is anchored on — reconciled against (1) in Silver. |
| 3 | NHS England RTT, July 2026 full CSV extract (10 Sep 2026) | Provider-level incomplete pathways by weekly wait band. |
| 4 | ONS Open Geography Portal lookups | LSOA → SICBL → ICB → NHS region → LAD (Apr 2026); OA → LSOA → MSOA; LAD → Region. |
| 5 | OHID, *NHS acute (hospital) trust catchment populations: April 2026* | The only published residence → provider crosswalk. Finding this changed the shape of the test. |

Files already on disk are not re-downloaded; the hash is recomputed either way.

In [2]:
run_step('01_bronze_ingest')

  exists   ons_hsle_msoa/hslemsoa.xlsx
  metadata ons_hsle_msoa/metadata.json
  exists   fingertips_hle_msoa/hle_94359_msoa.csv
  exists   fingertips_hle_msoa/indicator_94359_metadata.json
  metadata fingertips_hle_msoa/metadata.json
  exists   nhse_rtt_2026-07/Full-CSV-data-file-Jul26.zip
  metadata nhse_rtt_2026-07/metadata.json
  exists   ons_geography_lookups/LSOA21_SICBL26_ICB26_NHSER26_LAD26_EN_LU.csv
  exists   ons_geography_lookups/OA21_LSOA21_MSOA21_LAD23_EN_LU.csv
  exists   ons_geography_lookups/LAD24_RGN24_EN_LU.csv
  metadata ons_geography_lookups/metadata.json
  exists   ohid_trust_catchment_2026-04/nhs-acute-hospital-trust-catchment-populations-data_tables-april-2026.ods
  metadata ohid_trust_catchment_2026-04/metadata.json


<module '01_bronze_ingest' from '/Users/yusufismail/Two_maps/src/pipeline/01_bronze_ingest.py'>

## Step 2 — What landed, and its provenance

In [3]:
import json
rows = []
for meta in sorted(BRONZE.glob("*/metadata.json")):
    m = json.loads(meta.read_text())
    for f in m["files"]:
        rows.append({"source": meta.parent.name, "publisher": m["publisher"], "published": m["published_date"],
                     "file": f["filename"], "MB": round(f["bytes"] / 1e6, 2), "sha256": f["sha256"][:12] + "…"})
pd.DataFrame(rows)

,source,publisher,published,file,MB,sha256
0,fingertips_hle_msoa,Office for Health Improvement and Disparities,2026-09-02,hle_94359_msoa.csv,2.91,642b5e144630…
1,fingertips_hle_msoa,Office for Health Improvement and Disparities,2026-09-02,indicator_94359_metadata.json,0.00,64aad45522fd…
2,nhse_rtt_2026-07,NHS England,2026-09-10,Full-CSV-data-file-Jul26.zip,3.51,261d6a306827…
3,ohid_trust_catchment_2026-04,Office for Health Improvement and Disparities,2026-04-28,nhs-acute-hospital-trust-catchment-populations-data_tables-april-2026.ods,18.83,7f7639b2cc05…
4,ons_geography_lookups,Office for National Statistics (Open Geography Portal),2026-04 (ICB/LSOA lookup); 2023-04 (OA-LSOA-MSOA-LAD); 2024-12 (LAD-Region),LSOA21_SICBL26_ICB26_NHSER26_LAD26_EN_LU.csv,6.54,6cdd0cb55f2e…
5,ons_geography_lookups,Office for National Statistics (Open Geography Portal),2026-04 (ICB/LSOA lookup); 2023-04 (OA-LSOA-MSOA-LAD); 2024-12 (LAD-Region),OA21_LSOA21_MSOA21_LAD23_EN_LU.csv,21.93,302e4dc4b24f…
6,ons_geography_lookups,Office for National Statistics (Open Geography Portal),2026-04 (ICB/LSOA lookup); 2023-04 (OA-LSOA-MSOA-LAD); 2024-12 (LAD-Region),LAD24_RGN24_EN_LU.csv,0.01,73672e80f312…
7,ons_hsle_msoa,Office for National Statistics,2026-07-30,hslemsoa.xlsx,2.83,43ee369b7aef…


## Step 3 — A first look at each file, before any parsing decisions

Not cleaning anything here — just confirming the shapes match what the
publisher's documentation says, so that Silver's assertions are informed
rather than guessed.

In [4]:
ft = pd.read_csv(BRONZE / "fingertips_hle_msoa" / "hle_94359_msoa.csv")
print("Fingertips rows:", len(ft), "| area types:", ft["Area Type"].value_counts().to_dict())
ft[ft["Area Type"] == "England"][["Area Code", "Area Name", "Sex", "Time period", "Value", "Lower CI 95.0 limit", "Upper CI 95.0 limit"]].drop_duplicates()

Fingertips rows: 13716 | area types: {'MSOA': 13712, 'England': 4}


,Area Code,Area Name,Sex,Time period,Value,Lower CI 95.0 limit,Upper CI 95.0 limit
0,E92000001,England,Male,2019 - 23,65.1,65.1,65.1
1,E92000001,England,Female,2019 - 23,66.5,66.5,66.6


That England row — 65.1 male, 66.5 female — is "OHID's published England HLE"
that the reconciliation gate has to reproduce. Note the Fingertips area type is
labelled plain "Middle Super Output Area" (id 3), but the codes are 2021 MSOAs:
6,856 of them, which is the 2021 count for England.

In [5]:
x = pd.ExcelFile(BRONZE / "ons_hsle_msoa" / "hslemsoa.xlsx")
print("ONS sheets:", x.sheet_names)
ons2 = x.parse("2", header=6)
print("Sheet 2 rows:", len(ons2), "| area types:", ons2["Area type"].value_counts().to_dict())
x.parse("4", header=6)

ONS sheets: ['Cover_sheet', 'Contents', 'Methodology', 'Notes', '1', '2', '3', '4']
Sheet 2 rows: 15355 | area types: {'MSOA': 14527, 'Local Areas': 678, 'Integrated Care Board': 84, 'Combined Authority': 30, 'Region': 18, 'Welsh Health Board': 14, 'Country': 4}


,Period,Country,Area type,Area code,Area name,Sex,Sex code,LE,LCI,UCI,LE imputed
0,2019 to 2023,England,MSOA,E02006953,Barnet 042,Male,1,97.1,76.6,117.5,0


Sheet 4 is the one thing that would otherwise look like a bug downstream:
ONS excludes MSOAs whose *life expectancy* is implausible (their outer-fence
rule) and publishes no HLE for them. Exactly one MSOA-sex — Barnet 042, male,
LE 97.1 — so Silver will have 13,711 HLE values for 13,712 rows, and that is
correct, not a missing value.

In [6]:
cat = pd.ExcelFile(BRONZE / "ohid_trust_catchment_2026-04" / "nhs-acute-hospital-trust-catchment-populations-data_tables-april-2026.ods", engine="odf")
print("Catchment sheets:", cat.sheet_names)
t2 = cat.parse("All_admissions", header=2)
print("Table 2 rows:", len(t2), "| trusts:", t2["Trust code"].nunique(), "| MSOAs:", t2["MSOA21CD"].nunique())
t2.head(6)

Catchment sheets: ['Contents', 'Information', 'Trust_analysis', 'All_admissions', 'Elective', 'Emergency', 'Ethnicity', 'Deprivation', 'Trust_area_lookup', 'HES_years_covered']
Table 2 rows: 119613 | trusts: 134 | MSOAs: 6856


,Catchment year,Admission type,Trust code,Trust name,MSOA21CD,Patients admitted,Total patients,Patients admitted as a proportion of total patients,Trust total catchment population,MSOA21total catchment,First past the post (FPTP),Percentage admitted category
0,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000060,10,4295,0.002,1242363,31,False,Less than 5% not shown
1,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000984,200,2620,0.075,1242363,704,False,05% to 19%
2,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000985,250,2855,0.088,1242363,786,False,05% to 19%
3,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000986,230,3180,0.073,1242363,704,False,05% to 19%
4,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000987,215,3125,0.068,1242363,544,False,05% to 19%
5,2023,All Admissions,R0A,Manchester University NHS Foundation Trust,E02000988,280,3710,0.076,1242363,771,False,05% to 19%


This is the crosswalk the article's outline said didn't exist: MSOA of
residence × acute trust × patients admitted, with OHID's own "first past the
post" flag. It is admissions-based, acute-only, rounded to the nearest 5, and
many-to-many — all of which Gold measures. But it *exists*, which the outline
did not anticipate.

In [7]:
import zipfile
with zipfile.ZipFile(BRONZE / "nhse_rtt_2026-07" / "Full-CSV-data-file-Jul26.zip") as z:
    name = [n for n in z.namelist() if n.endswith(".csv")][0]
    rtt = pd.read_csv(z.open(name), usecols=["Period", "Provider Parent Org Code", "Provider Parent Name", "Provider Org Code",
                                             "Commissioner Org Code", "RTT Part Description", "Treatment Function Code", "Total", "Total All"], low_memory=False)
print("RTT rows:", len(rtt), "| parts:", rtt["RTT Part Description"].value_counts().to_dict())
inc = rtt[rtt["RTT Part Description"] == "Incomplete Pathways"]
print("Incomplete: providers", inc["Provider Org Code"].nunique(), "| provider parents (ICBs)", inc["Provider Parent Org Code"].nunique(),
      "| 'Total' populated:", inc["Total"].notna().mean(), "| 'Total All' populated:", inc["Total All"].notna().mean())
inc[["Provider Parent Org Code", "Provider Parent Name"]].drop_duplicates().sort_values("Provider Parent Org Code").reset_index(drop=True)

RTT rows: 182372 | parts: {'Incomplete Pathways': 63223, 'New RTT Periods - All Patients': 37524, 'Completed Pathways For Non-Admitted Patients': 32157, 'Incomplete Pathways with DTA': 30441, 'Completed Pathways For Admitted Patients': 19027}
Incomplete: providers 533 | provider parents (ICBs) 36 | 'Total' populated: 0.0 | 'Total All' populated: 1.0


,Provider Parent Org Code,Provider Parent Name
0,D7T5G,NHS ESSEX INTEGRATED CARE BOARD
1,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE BOARD
2,QF7,NHS SOUTH YORKSHIRE INTEGRATED CARE BOARD
3,QGH,NHS HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CARE BOARD
4,QHL,NHS BIRMINGHAM AND SOLIHULL INTEGRATED CARE BOARD
5,QHM,NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CARE BOARD
6,QJ2,NHS DERBY AND DERBYSHIRE INTEGRATED CARE BOARD
7,QJK,NHS DEVON INTEGRATED CARE BOARD
8,QJM,NHS LINCOLNSHIRE INTEGRATED CARE BOARD
9,QK1,"NHS LEICESTER, LEICESTERSHIRE AND RUTLAND INTEGRATED CARE BOARD"


Two things worth noticing before Silver: the `Total` column is blank for every
incomplete-pathway row (only `Total All` is populated — Silver rebuilds totals
from the wait bands and asserts they match), and there are 36 provider parents,
not 42. ICBs were reorganised in April 2026. The ONS April 2026 lookup has the
same 36, which is what makes the ICB-level join clean at the key level — the
objection the article has to answer with numbers rather than keys.